In [12]:
from pathlib import Path
import polars as pl

import numpy as np
from scipy.stats import spearmanr, kendalltau

In [13]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where group folder should be: {BASE}")
print(f"Current working directory (where table will be saved): {CWD}")

Folder where group folder should be: /data/users/bdupin/datasets-organelle-igr
Current working directory (where table will be saved): /data/users/bdupin/datasets-organelle-igr/code


In [14]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]

label_map = {
    "fungi_mit": "Fungi (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_plt": "Green algae (plastid)",
    "plants_plt": "Plants (plastid)",
    "protists_plt": "Protists (plastid)",
}

In [15]:
name_map = {
    "m_both": "Polarity + length + type",
    "m_type": "Polarity + flanking-gene type",
    "m_len" : "Polarity + maximum flanking-gene length"}

def parse_kfold_block(text):
    lines = text.splitlines()
    in_block = False
    out = {}
 
    for line in lines:
        ls = line.strip()
        if ls.startswith("--- K-fold"):
            in_block = True
            continue
        if in_block:
            if ls.startswith("elpd_diff"):
                continue  # header row
            parts = ls.split()
            if len(parts) == 3 and parts[0] in name_map:
                model, elpd_diff, se_diff = parts
                out[name_map[model]] = (float(elpd_diff), float(se_diff))
            elif ls == "" or ls.startswith("Interpreting"):
                break
 
    if len(out) != 3:
        raise ValueError(f"Expected 3 models in K-fold block, found {len(out)}: {out}")
    return out

In [17]:
rows_s2 = []

model_key_map = {
    "gene_length": "Polarity + maximum flanking-gene length", 
    "gene_type": "Polarity + flanking-gene type", 
    "both": "Polarity + length + type",
    }

for g in groups:
    gdir = BASE / g
    
    brms_pol = gdir / "brms_polarity" / "brms_results_row.tsv"
    brms_type_length_dir = gdir / "brms_type_length"
    brms_type_length = brms_type_length_dir / "brms_results_row.tsv"
    brms_type_length_txt = brms_type_length_dir / "brms_result.txt"
 
    tsv_polarity = pl.read_csv(brms_pol, separator="\t")
    tsv_type_length = pl.read_csv(brms_type_length, separator="\t")
    kfold = parse_kfold_block(brms_type_length_txt.read_text())
 
    label = label_map[g]
 
    # ---- base (polarity-only) model: single row ----
    polarity = tsv_polarity.row(0, named=True)
    rows_s2.append(
        {
            "group": label,
            "model": "Polarity only",
            "N_igr": polarity["N_regions"],
            "N_taxa": polarity["N_taxa"],
            "fold_conv": polarity["fold_convergent_over_same"],
            "fold_conv_lo": polarity["fold_convergent_lo"],
            "fold_conv_hi": polarity["fold_convergent_hi"],
            "fold_div": polarity["fold_divergent_over_same"],
            "fold_div_lo": polarity["fold_divergent_lo"],
            "fold_div_hi": polarity["fold_divergent_hi"],
            "Difference in expected log predictive density": None,
            "SE of ELPD difference": None,
        }
    )
 
    # ---- len / type / both models: one row each ----
    for pred_type, model_name in model_key_map.items():
        r = tsv_type_length.filter(pl.col("predictor_type") == pred_type).row(0, named=True)
        elpd_diff, se_diff = kfold[model_name]
        rows_s2.append(
            {
                "group": label,
                "model": model_name,
                "N_igr": r["N_regions"],
                "N_taxa": r["N_taxa"],
                "fold_conv": r["fold_convergent_over_same"],
                "fold_conv_lo": r["fold_convergent_lo"],
                "fold_conv_hi": r["fold_convergent_hi"],
                "fold_div": r["fold_divergent_over_same"],
                "fold_div_lo": r["fold_divergent_lo"],
                "fold_div_hi": r["fold_divergent_hi"],
                "Difference in expected log predictive density": elpd_diff,
                "SE of ELPD difference": se_diff,
            }
        )
 
s2 = pl.DataFrame(rows_s2)

float_cols = [col for col in s2.columns if s2[col].dtype in [pl.Float32, pl.Float64]]
s2 = s2.with_columns([pl.col(col).round(2) for col in float_cols])

s2.write_csv(BASE / "code" / "supplemental_table2.tsv", separator="\t")